<a href="https://colab.research.google.com/github/Harinicode/Mini_project/blob/main/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
from google.colab import drive
drive.mount('/content/drive')


MessageError: Error: credential propagation was unsuccessful

In [8]:
!pip install pandas pyarrow opencv-python matplotlib

In [13]:
from google.colab import files
files.upload()  # Select your kaggle.json file when prompted


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"harini234444444444","key":"5418830b0a15974ba116691ed8e2af5e"}'}

In [14]:
import os

# Move kaggle.json to the correct directory
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [15]:
# Create dataset directory if it doesn’t exist
os.makedirs("dataset", exist_ok=True)

# Download dataset from Kaggle
!kaggle datasets download -d borhanitrash/alzheimer-mri-disease-classification-dataset -p dataset

# Unzip dataset
!unzip dataset/alzheimer-mri-disease-classification-dataset.zip -d dataset


Dataset URL: https://www.kaggle.com/datasets/borhanitrash/alzheimer-mri-disease-classification-dataset
License(s): apache-2.0
 89% 23.0M/26.0M [00:00<00:00, 121MB/s] 
100% 26.0M/26.0M [00:00<00:00, 122MB/s]
Archive:  dataset/alzheimer-mri-disease-classification-dataset.zip
  inflating: dataset/Alzheimer MRI Disease Classification Dataset/Data/test-00000-of-00001-44110b9df98c5585.parquet  
  inflating: dataset/Alzheimer MRI Disease Classification Dataset/Data/train-00000-of-00001-c08a401c53fe5312.parquet  
  inflating: dataset/Alzheimer MRI Disease Classification Dataset/README.md  


In [16]:
!ls -R dataset


dataset:
'Alzheimer MRI Disease Classification Dataset'	 alzheimer-mri-disease-classification-dataset.zip

'dataset/Alzheimer MRI Disease Classification Dataset':
Data  README.md

'dataset/Alzheimer MRI Disease Classification Dataset/Data':
test-00000-of-00001-44110b9df98c5585.parquet  train-00000-of-00001-c08a401c53fe5312.parquet


In [17]:
import pandas as pd

# Load train and test datasets
train_df = pd.read_parquet("dataset/Alzheimer MRI Disease Classification Dataset/Data/train-00000-of-00001-c08a401c53fe5312.parquet")
test_df = pd.read_parquet("dataset/Alzheimer MRI Disease Classification Dataset/Data/test-00000-of-00001-44110b9df98c5585.parquet")

# Check dataset structure
print(train_df.head())  # Preview first few rows
print(train_df.columns)  # Check available columns


                                               image  label
0  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      2
1  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      0
2  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      3
3  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      3
4  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...      2
Index(['image', 'label'], dtype='object')


In [10]:
!ls -R


.:
sample_data

./sample_data:
anscombe.json		     california_housing_train.csv  mnist_train_small.csv
california_housing_test.csv  mnist_test.csv		   README.md


In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Function to convert binary image data to a NumPy array
def convert_image(image_dict):
    binary_data = image_dict["bytes"]  # Extract the 'bytes' field from the dictionary
    img_array = np.frombuffer(binary_data, dtype=np.uint8)  # Convert bytes to NumPy array
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)  # Decode as an image
    return img

# Convert and display the first sample image
sample_img = convert_image(train_df.iloc[0]["image"])  # Use 'image' column
plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


In [ ]:
IMG_SIZE = 128  # Set image size

def preprocess_image(image_dict):
    img = convert_image(image_dict)  # Convert binary to image
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))  # Resize to 128x128
    img = img / 255.0  # Normalize pixel values
    return img

# Apply preprocessing to the dataset
train_df['processed_image'] = train_df['image'].apply(preprocess_image)
test_df['processed_image'] = test_df['image'].apply(preprocess_image)


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt

# Define augmentation parameters

datagen = ImageDataGenerator(
   # rotation_range=10,     # Rotate images up to 20 degrees
    zoom_range=0.2,        # Randomly zoom in/out
    horizontal_flip=True,  # Flip images horizontally
   # brightness_range=[0.8, 1.2]  # Adjust brightness
)

# Convert processed images into NumPy arrays
train_images = np.stack(train_df["processed_image"].values)
test_images = np.stack(test_df["processed_image"].values)

# Expand dimensions to match CNN input (batch, height, width, channels)
train_images = np.expand_dims(train_images, axis=-1)  # Adding grayscale channel
test_images = np.expand_dims(test_images, axis=-1)

# Apply augmentation to training images
augmented_images = []
for img in train_images:
    img = img.squeeze()  # Remove extra dimensions if any

    # Ensure grayscale image is correctly converted to RGB
    if img.ndim == 2:
        img_rgb = np.stack([img] * 3, axis=-1)  # Convert grayscale (128,128) -> (128,128,3)
    elif img.shape[-1] == 1:
        img_rgb = np.repeat(img, 3, axis=-1)  # Convert (128,128,1) -> (128,128,3)
    else:
        img_rgb = img  # Already in correct shape

    augmented_img = datagen.random_transform(img_rgb)  # Apply augmentation
    augmented_images.append(augmented_img)

# Convert to NumPy array
augmented_images = np.array(augmented_images)



In [ ]:
# Verify augmentation by displaying a few images
plt.figure(figsize=(10, 5))
for i in range(5):
    plt.subplot(1, 5, i+1)

    # Convert image to uint8 format for proper display
    img_display = (augmented_images[i] * 255).astype(np.uint8)

    # Display the image correctly
    plt.imshow(img_display.squeeze(), cmap="gray")
    plt.axis("off")

plt.show()


In [ ]:
print("Min pixel value:", np.min(augmented_images))
print("Max pixel value:", np.max(augmented_images))
